# Vectors and Vector Spaces

| Difficulty | ██░░░░░░░░ 2/10 |
| :--- | :--- |
| Prerequisites | 00.01–00.04 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/01_vectors_and_spaces.ipynb)

---

## 🎯 Learning Objective

Understand vectors as objects that live in vector spaces, and learn the norms and similarity measures used to compare them.

---

## 📐 Theory

A **vector** has two equivalent faces: algebraically, it's a list of numbers
$\mathbf{x} = (x_1, \dots, x_n)$; geometrically, it's an arrow from the origin to a point in
space. Both views matter — the algebra is what you compute with, the geometry is what
you should *picture* while computing.

### The dot product

The dot product combines two vectors into a single number:

$$\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^{n} u_i v_i = \|\mathbf{u}\|\,\|\mathbf{v}\|\cos\theta$$

The second equality is the geometric payoff: the dot product encodes *both* how long the
vectors are *and* the angle $\theta$ between them.

### Norms: measuring "size"

A norm turns a vector into a single non-negative number representing its magnitude. Three
show up constantly in ML:

$$\|\mathbf{x}\|_1 = \sum_i |x_i| \qquad
\|\mathbf{x}\|_2 = \sqrt{\sum_i x_i^2} \qquad
\|\mathbf{x}\|_\infty = \max_i |x_i|$$

$L^1$ ("Manhattan") norm sums absolute values — it's robust to outliers and encourages
*sparsity* when used as a regularizer. $L^2$ ("Euclidean") norm is straight-line distance —
the default notion of length. $L^\infty$ only cares about the single largest entry.

### Cosine similarity

Dividing out the magnitudes from the dot product isolates pure *direction*:

$$\cos\theta = \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}$$

This ranges from $-1$ (opposite directions) to $1$ (same direction), completely ignoring how
long each vector is — which is exactly why it's the standard similarity measure for embeddings
that can vary wildly in magnitude.

### Vector spaces and subspaces

A **vector space** is a set closed under vector addition and scalar multiplication — combine
any two members (or scale one) and you stay inside the set. A **subspace** is a subset of a
vector space that is itself a vector space (in particular, it must contain the origin). The
**span** of a set of vectors is the subspace of all their possible linear combinations, and a
set of vectors is **linearly independent** if none of them can be written as a combination of
the others — each one adds a genuinely new direction.

---

## 👁️ Visual Intuition

Plotting two vectors as arrows from the origin makes the dot product's geometric meaning
concrete: the norms are the arrow lengths, and $\cos\theta$ is read directly off the angle
between them.

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

u = np.array([3.0, 1.0])
v = np.array([1.0, 2.0])
cos_theta = (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))

fig, ax = plt.subplots(figsize=(5, 5))
ax.quiver(0, 0, u[0], u[1], angles='xy', scale_units='xy', scale=1, color="#4C72B0", width=0.012, label="u")
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color="#DD8452", width=0.012, label="v")
ax.set_xlim(-1, 4); ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.legend()
ax.set_title(f"||u||={np.linalg.norm(u):.2f}   ||v||={np.linalg.norm(v):.2f}   cos(theta)={cos_theta:.2f}")
plt.show()

## 🐍 Implementation from Scratch

Every quantity from the theory section is a one-line loop. Implementing them by hand first,
then checking against NumPy, builds the muscle memory to read vectorized code fluently.

In [ ]:
# Dot product, norms, and cosine similarity -- pure Python, then cross-checked with NumPy
import numpy as np

def dot(u, v):
    return sum(ui * vi for ui, vi in zip(u, v))

def l1_norm(x):
    return sum(abs(xi) for xi in x)

def l2_norm(x):
    return sum(xi ** 2 for xi in x) ** 0.5

def linf_norm(x):
    return max(abs(xi) for xi in x)

def cosine_similarity(u, v):
    return dot(u, v) / (l2_norm(u) * l2_norm(v))

rng = np.random.default_rng(0)
a, b = rng.standard_normal(6), rng.standard_normal(6)

print("dot(a,b)          :", round(dot(a, b), 4), " vs np.dot:", round(float(a @ b), 4))
print("L1 norm of a       :", round(l1_norm(a), 4), " vs numpy:", round(float(np.linalg.norm(a, 1)), 4))
print("L2 norm of a       :", round(l2_norm(a), 4), " vs numpy:", round(float(np.linalg.norm(a, 2)), 4))
print("Linf norm of a     :", round(linf_norm(a), 4), " vs numpy:", round(float(np.linalg.norm(a, np.inf)), 4))
print("cosine_similarity  :", round(cosine_similarity(a, b), 4))

## 🤖 Why This Matters for AI

**Word embeddings** map words to vectors such that geometric relationships mirror semantic
ones. **Cosine similarity** is the standard way to compare them — because it ignores
magnitude, two embeddings pointing the same direction are "similar" regardless of how
confidently-scaled either one happens to be. This is the exact computation behind semantic
search and retrieval-augmented generation: embed the query, embed every document, rank by
cosine similarity.

Below, we hand-craft toy embeddings where `queen = king - man + woman` *exactly*, to make the
famous word2vec-style analogy concrete and verifiable rather than mysterious.

In [ ]:
# Cosine similarity powering semantic search + the classic word-vector analogy
import numpy as np

# Toy 4D "embeddings" -- in a real model these are learned, but here we hand-craft them so the
# geometric relationships are exact and easy to verify by hand.
embeddings = {
    "king":   np.array([0.80, 0.90, 0.10, 0.00]),
    "man":    np.array([0.60, 0.10, 0.10, 0.00]),
    "woman":  np.array([0.60, 0.90, 0.10, 0.00]),
    "queen":  np.array([0.80, 1.70, 0.10, 0.00]),   # == king - man + woman, by construction
    "apple":  np.array([0.00, 0.00, 0.90, 0.70]),
    "orange": np.array([0.00, 0.10, 0.80, 0.90]),
}

def cosine_similarity(u, v):
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))

# --- Semantic search: rank every word by similarity to a "fruit-like" query vector ---
query = np.array([0.05, 0.05, 0.85, 0.75])
ranked = sorted(embeddings.items(), key=lambda kv: cosine_similarity(query, kv[1]), reverse=True)
print("Semantic search results (most similar to query first):")
for word, vec in ranked:
    print(f"  {word:8s} cosine similarity = {cosine_similarity(query, vec):.3f}")

# --- Analogy: king - man + woman =~ queen ---
analogy_vector = embeddings["king"] - embeddings["man"] + embeddings["woman"]
candidates = {w: cosine_similarity(analogy_vector, e) for w, e in embeddings.items() if w not in ("king", "man", "woman")}
best_word, best_sim = max(candidates.items(), key=lambda kv: kv[1])
print(f"\n'king - man + woman' is most similar to: '{best_word}'  (cosine similarity={best_sim:.3f})")

## 🏋️ Exercises

### Warm-up
1. For $\mathbf{x} = (3, -4, 0, 1)$, compute $\|\mathbf{x}\|_1$, $\|\mathbf{x}\|_2$, and $\|\mathbf{x}\|_\infty$ by hand, then verify with `l1_norm`/`l2_norm`/`linf_norm`.

### Practice
2. Generate 5 random vectors in $\mathbb{R}^4$ and use `cosine_similarity` to find the most similar pair and the most dissimilar (most negative) pair.

### Challenge
3. Extend the toy `embeddings` dictionary with 3 new words of your own that satisfy an analogy of your choosing (e.g. a country/capital relationship), verify with cosine similarity that the analogy vector's nearest neighbor is the word you intended. Then multiply just the `queen` vector by 10 (change its magnitude only) and confirm cosine similarity to the analogy vector is UNCHANGED -- explain in one sentence why this magnitude-invariance is desirable for comparing embeddings of words that appear with very different frequencies during training.

---

## 📚 Further Reading
- Mikolov et al., ["Efficient Estimation of Word Representations in Vector Space"](https://arxiv.org/abs/1301.3781) (word2vec)
- Deisenroth, Faisal & Ong, *Mathematics for Machine Learning*, Ch. 3 (Analytic Geometry)

---

*Next notebook: [Matrix Operations](02_matrix_operations.ipynb)*